In [1]:
import lzma
import os
import pickle
from itertools import product
from pathlib import Path

import fire
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data
from tqdm import tqdm
from torchmetrics.classification import BinaryF1Score

from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from fast_boolean_analysis import FourierSeries, fourier_expand
from pytorchtools import EarlyStopping
from spin_lattices import KagomeLattice, SpinLattice, SquareLattice
from spin_nn import FC1SpinNN, SpinNN
from utils import make_unpacked_configurations, get_abslargest_terms
from lattice_boolean_analysis import (
    SignSignalKind,
    AmplitudeMedianBinSignalKind,
    LBFFromNN,
    LBFFromSpinSystem,
)
from loguru import logger
import numpy.typing as npt
from utils import hadamard_transform_pytorch_inplace

self_name = "multi_nn_2023_02_27.py"


/vol/tcm10/ischurov/.conda/envs/latsym2/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: /vol/tcm10/ischurov/.conda/envs/latsym2/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs
  warn(f"Failed to load image Python extension: {e}")
2023-03-03 18:14:13.165 | DEBUG    | lattice_symmetries:__init__:49 - Initializing Haskell runtime...
2023-03-03 18:14:13.166 | DEBUG    | lattice_symmetries:__init__:51 - Initializing Chapel runtime...
[Debug]   [2023-03-03 18:14:13.205 | DEBUG    | lattice_symmetries:__init__:53 - Setting Python exception handler...
LOCALE0]   Initializing chpl_kernels ...
set_python_exception_handler ...


In [2]:
ground_state_cache_dir = Path("groundstates")

experiment_dir = Path("experiments") / self_name.removesuffix(".py")
nn_checkpoints_dir = experiment_dir / "nn_checkpoints"


In [3]:
lattices: list[SpinLattice] = [SquareLattice(width=5, height=6), KagomeLattice(width=2, height=4)]
signal_kind = SignSignalKind()

target_scorer = "accuracy"
target_score = 0.8

eps_trains = [1e-3, 5e-2, 1e-2]
val_eps = 5e-2
test_eps = 5e-2
epochs = 20000
patience = 1
delta = 0.01
batch_size = 64

J2s = [0.1]


In [4]:
lattice = lattices[0]
J2 = J2s[0]
eps_train = eps_trains[0]
system = HeisenbergJ1J2(
    lattice=lattice,
    J1=1,
    J2=J2,
    ground_state_cache_dir=ground_state_cache_dir,
)
system.get_eigenstates(1)

model_path = (
    nn_checkpoints_dir / f"FC1-1hidden-64-sign-{system.get_cache_id()}_eps_train={eps_train}.pt"
)
net = nn.Sequential(
    nn.Linear(system.number_spins, 64, dtype=torch.float64),
    nn.ReLU(),
    nn.Linear(64, 2, dtype=torch.float64),
)


2023-03-03 18:14:16.095 | DEBUG    | heisenberg_hamiltonians:__init__:433 - use_symmetries is None and lattice is not in symmetries whitelist, setting use_symmetries=False
2023-03-03 18:14:16.097 | DEBUG    | heisenberg_hamiltonians:__init__:448 - number_spins=30
2023-03-03 18:14:16.099 | DEBUG    | heisenberg_hamiltonians:__init__:458 - Symmetry group contains 0 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-03-03 18:14:16.988 | DEBUG    | heisenberg_hamiltonians:__init__:467 - Hilbert space dimension is 155117520
2023-03-03 18:14:17.016 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:60 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-SquareLattice5x6-1.0-0.1-False-None-1.pickle
2023-03-03 18:14:18.506 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:107 - Ground state energy is -73.5944058493


In [5]:
net.load_state_dict(torch.load(model_path))


<All keys matched successfully>

In [6]:
def get_probs(system: SpinSystem):
    logger.debug("get_df_ground_state")
    gs_df = system.get_df_ground_state(canonical_basis=True, add_amplitude=False)
    logger.debug("Finding probabilities")
    gs_df["prob"] = np.abs(gs_df["eigenstate_coeff"]) ** 2
    logger.debug("Keeping only probabilities")
    prob = gs_df["prob"]
    return prob


In [7]:
nn_signal = LBFFromNN(
    lattice=lattice,
    nn=net,
    probs=get_probs(system),
)


2023-03-03 18:14:21.190 | DEBUG    | __main__:get_probs:2 - get_df_ground_state
2023-03-03 18:14:21.645 | DEBUG    | __main__:get_probs:4 - Finding probabilities
2023-03-03 18:14:22.860 | DEBUG    | __main__:get_probs:6 - Keeping only probabilities


In [11]:
signal = nn_signal
x = signal.canonical_basis.states
logger.debug("Finding signal")
signal_value = signal.as_long_array(x).copy()


2023-03-03 18:16:26.903 | DEBUG    | __main__:<module>:3 - Finding signal


In [12]:
signal_value.shape

(1073741824,)

In [13]:
tensor = torch.tensor(signal_value, dtype=torch.float64)

In [14]:
(tensor.element_size() * tensor.nelement()) / ((2 ** 10) ** 3)

8.0

In [15]:
transformed_tensor = hadamard_transform_pytorch_inplace(tensor)

In [16]:
series = FourierSeries(
    signal=signal,
    coeffs=transformed_tensor.numpy(),
)

In [17]:
series.how_many_terms_to_achieve_score(0.9, 'accuracy')

2023-03-03 18:20:24.678 | DEBUG    | fast_boolean_analysis:how_many_terms_to_achieve_score:148 - At min_terms=1, score=1.0 >= target_score=0.9, so we can achieve the target score with 1 terms


(True,
 1,
 array([0.14446445, 0.14446445, 0.14446445, ..., 0.14446445, 0.14446445,
        0.14446445]))

In [18]:
series.total_hamming_weight(1)

0